# Ablation Study: ECG Route Comparison

Compare evaluation metrics across the three ECG routes:
- **no_ecg** — baseline, no ECG tokens
- **fusion_class** — single fused ECG classification token
- **all_branches** — full multi-branch ECG prototype tokens

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

In [ ]:
# ── Configure ────────────────────────────────────────────────────
# Adjust these to point at your evaluation outputs
SIZE = "tiny"
N_PATIENTS = 10000  # set to None for full run

BASE = Path("../data/processed")
if N_PATIENTS is not None:
    BASE = BASE / f"n{N_PATIENTS}"

ROUTES = ["no_ecg", "fusion_class", "all_branches"]
ROUTE_LABELS = {"no_ecg": "No ECG", "fusion_class": "Fusion Class", "all_branches": "All Branches"}
ROUTE_COLORS = {"no_ecg": "#7f8c8d", "fusion_class": "#e67e22", "all_branches": "#2980b9"}

# Minimum positives to include a label (avoids noisy metrics)
MIN_POSITIVES = 20

In [ ]:
# ── Load metrics ────────────────────────────────────────────────
raw: dict[str, dict] = {}
for route in ROUTES:
    path = BASE / "evaluation" / f"{route}_{SIZE}" / "metrics.json"
    if not path.exists():
        print(f"WARNING: missing {path}")
        continue
    with open(path) as f:
        raw[route] = json.load(f)
    print(f"Loaded {route}: {len(raw[route])} label-horizon entries")

assert len(raw) == len(ROUTES), f"Missing routes: {set(ROUTES) - set(raw)}"

In [ ]:
# ── Build tidy DataFrame ────────────────────────────────────────
rows = []
for route, metrics in raw.items():
    for key, vals in metrics.items():
        # key format: "LABEL//label_name_Xh"
        # Split from the right: last part is horizon like "8h", "24h", "48h"
        label_horizon = key.replace("LABEL//", "")
        parts = label_horizon.rsplit("_", 1)
        if len(parts) != 2:
            continue
        label_name, horizon_str = parts
        horizon = int(horizon_str.replace("h", ""))
        rows.append({
            "route": route,
            "label": label_name,
            "horizon": horizon,
            "auroc": vals.get("auroc"),
            "auprc": vals.get("auprc"),
            "brier": vals.get("brier"),
            "ece": vals.get("ece"),
            "n_positive": vals.get("n_positive", 0),
            "n_total": vals.get("n_total", 0),
        })

df = pd.DataFrame(rows)
print(f"{len(df)} rows: {df['label'].nunique()} labels x {df['horizon'].nunique()} horizons x {df['route'].nunique()} routes")
df.head()

In [ ]:
# ── Filter to evaluable labels ──────────────────────────────────
# Drop labels with too few positives or null AUROC
df_eval = df[(df["n_positive"] >= MIN_POSITIVES) & df["auroc"].notna()].copy()
kept_labels = sorted(df_eval["label"].unique())
print(f"Evaluable labels (n_positive >= {MIN_POSITIVES}): {len(kept_labels)}")
print(", ".join(kept_labels))

## 1. AUROC Comparison — Grouped Bar Chart

In [ ]:
def plot_grouped_bars(df_plot, metric, title, horizons=None):
    """Grouped bar chart: one group per label, one bar per route."""
    if horizons is None:
        horizons = sorted(df_plot["horizon"].unique())

    fig, axes = plt.subplots(1, len(horizons), figsize=(7 * len(horizons), 6), sharey=True)
    if len(horizons) == 1:
        axes = [axes]

    for ax, h in zip(axes, horizons):
        sub = df_plot[df_plot["horizon"] == h].copy()
        pivot = sub.pivot(index="label", columns="route", values=metric)
        # Reorder routes
        pivot = pivot[[r for r in ROUTES if r in pivot.columns]]
        # Sort by no_ecg value (baseline)
        if "no_ecg" in pivot.columns:
            pivot = pivot.sort_values("no_ecg", ascending=True)

        x = np.arange(len(pivot))
        width = 0.25
        for i, route in enumerate(pivot.columns):
            ax.barh(
                x + i * width,
                pivot[route],
                height=width,
                label=ROUTE_LABELS.get(route, route),
                color=ROUTE_COLORS.get(route, None),
                alpha=0.85,
            )

        ax.set_yticks(x + width)
        ax.set_yticklabels(pivot.index, fontsize=9)
        ax.set_xlabel(metric.upper())
        ax.set_title(f"{h}h horizon")
        ax.legend(loc="lower right", fontsize=8)
        if metric == "auroc":
            ax.axvline(0.5, color="red", ls="--", lw=0.8, alpha=0.5)

    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()
    plt.show()


plot_grouped_bars(df_eval, "auroc", "AUROC by Label and ECG Route")

## 2. AUROC Delta vs Baseline (no_ecg)

In [ ]:
# Compute delta AUROC relative to no_ecg baseline
baseline = df_eval[df_eval["route"] == "no_ecg"][["label", "horizon", "auroc"]].rename(
    columns={"auroc": "auroc_baseline"}
)
df_delta = df_eval[df_eval["route"] != "no_ecg"].merge(baseline, on=["label", "horizon"])
df_delta["delta_auroc"] = df_delta["auroc"] - df_delta["auroc_baseline"]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
for ax, h in zip(axes, sorted(df_delta["horizon"].unique())):
    sub = df_delta[df_delta["horizon"] == h]
    for route in ["fusion_class", "all_branches"]:
        rsub = sub[sub["route"] == route].sort_values("delta_auroc")
        colors = ["#27ae60" if d > 0 else "#c0392b" for d in rsub["delta_auroc"]]
        ax.barh(
            rsub["label"] + f" ({ROUTE_LABELS[route][:3]})",
            rsub["delta_auroc"],
            color=colors,
            alpha=0.7,
            label=ROUTE_LABELS[route],
        )

    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("\u0394 AUROC vs no_ecg")
    ax.set_title(f"{h}h horizon")
    ax.legend(fontsize=8)

fig.suptitle("AUROC Improvement over Baseline (no_ecg)", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## 3. Summary Table — Mean Metrics by Route

In [ ]:
summary = (
    df_eval.groupby(["route", "horizon"])
    .agg(
        mean_auroc=("auroc", "mean"),
        mean_auprc=("auprc", "mean"),
        mean_brier=("brier", "mean"),
        n_labels=("label", "nunique"),
    )
    .round(4)
    .reset_index()
)
summary["route"] = summary["route"].map(ROUTE_LABELS)
summary = summary.sort_values(["horizon", "route"])
summary

## 4. Heatmap — AUROC by Label x Route (24h horizon)

In [ ]:
def plot_heatmap(df_plot, horizon, metric="auroc"):
    sub = df_plot[df_plot["horizon"] == horizon]
    pivot = sub.pivot(index="label", columns="route", values=metric)
    pivot = pivot[[r for r in ROUTES if r in pivot.columns]]
    pivot.columns = [ROUTE_LABELS.get(c, c) for c in pivot.columns]
    pivot = pivot.sort_values(pivot.columns[0], ascending=False)

    fig, ax = plt.subplots(figsize=(6, max(4, len(pivot) * 0.35)))
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0.4, vmax=1.0)

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=10)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)

    # Annotate cells
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8,
                        color="white" if val < 0.55 else "black")

    fig.colorbar(im, ax=ax, shrink=0.8, label=metric.upper())
    ax.set_title(f"{metric.upper()} — {horizon}h horizon", fontsize=13, fontweight="bold")
    fig.tight_layout()
    plt.show()


plot_heatmap(df_eval, 24)

## 5. Radar Chart — Top Labels

In [ ]:
# Pick top labels by mean AUROC across routes at 24h
sub_24 = df_eval[df_eval["horizon"] == 24]
top_labels = (
    sub_24.groupby("label")["auroc"].mean()
    .sort_values(ascending=False)
    .head(8)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"projection": "polar"})
angles = np.linspace(0, 2 * np.pi, len(top_labels), endpoint=False).tolist()
angles += angles[:1]  # close polygon

for route in ROUTES:
    rsub = sub_24[sub_24["route"] == route].set_index("label")
    values = [rsub.loc[l, "auroc"] if l in rsub.index else 0.5 for l in top_labels]
    values += values[:1]
    ax.plot(angles, values, "o-", label=ROUTE_LABELS[route], color=ROUTE_COLORS[route], linewidth=2)
    ax.fill(angles, values, alpha=0.1, color=ROUTE_COLORS[route])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(top_labels, fontsize=8)
ax.set_ylim(0.4, 1.0)
ax.set_title("Top Labels — AUROC at 24h", fontsize=13, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

## 6. Prevalence vs Performance

In [ ]:
sub_24 = df_eval[df_eval["horizon"] == 24].copy()
sub_24["prevalence"] = sub_24["n_positive"] / sub_24["n_total"]

fig, ax = plt.subplots(figsize=(10, 6))
for route in ROUTES:
    rsub = sub_24[sub_24["route"] == route]
    ax.scatter(
        rsub["prevalence"], rsub["auroc"],
        label=ROUTE_LABELS[route], color=ROUTE_COLORS[route],
        s=60, alpha=0.7, edgecolors="white", linewidth=0.5,
    )

ax.axhline(0.5, color="red", ls="--", lw=0.8, alpha=0.5, label="Chance")
ax.set_xlabel("Label Prevalence", fontsize=11)
ax.set_ylabel("AUROC", fontsize=11)
ax.set_title("Prevalence vs AUROC — 24h horizon", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 7. Full Metrics Table

In [ ]:
# Pivot: label x horizon, showing AUROC for each route side by side
pivot = df_eval.pivot_table(
    index=["label", "horizon"],
    columns="route",
    values="auroc",
).round(4)
pivot.columns = [ROUTE_LABELS.get(c, c) for c in pivot.columns]

# Add delta columns
if "No ECG" in pivot.columns:
    for col in ["Fusion Class", "All Branches"]:
        if col in pivot.columns:
            pivot[f"\u0394 {col}"] = (pivot[col] - pivot["No ECG"]).round(4)

# Add n_positive for context
n_pos = df_eval[df_eval["route"] == "no_ecg"].set_index(["label", "horizon"])["n_positive"]
pivot["n_pos"] = n_pos

with pd.option_context("display.max_rows", 200):
    display(pivot.sort_index())